# Chapter 12: Dataset and DataLoader

[Read this chapter online](https://jackluu.io/book/section-4-training/ch12-dataset-and-dataloader/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch12-dataset-and-dataloader.ipynb)

From *Teach Your Computer to Write: Build and Train an LLM from Zero* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 12: Dataset and DataLoader

![The map with Training highlighted](https://jackluu.io/book/assets/diagrams/ch12-where-we-are.png)

*Figure 12.1: We begin the training module by preparing our data pipeline.*

So we have our model. From Chapter 11 we know we want to predict the next token. Time to input data into our model. We are not going to enter individual characters into our model but instead input thousands of examples at once. This chapter will be used to create a pipeline to batch our Shakespeare dataset.

In this chapter you will:

- Use a sliding window to generate training examples.
- Understand how batches process multiple sequences in parallel.
- Build a PyTorch Dataset and DataLoader.

> **Words to Know**
>
> - **Block Size**: The maximum number of tokens the model can look at at one time (its context window).
> - **Batching**: Grouping multiple training examples together and processing them at the same time.
> - **Dataset**: A PyTorch tool that defines how to fetch one training example from the text.
> - **DataLoader**: A PyTorch tool that automatically groups single examples into batches and shuffles them.

## Theory

### The Sliding Window

In the previous chapter, we learned how one sequence can generate many training examples. But where do we get those sequences from? For instance, how do we get sequences from our large text file of Shakespeare plays?

We use a sliding window (Figure 12.2).

![A sliding window over text](https://jackluu.io/book/assets/diagrams/ch12-sliding-window.png)

*Figure 12.2: A sliding window creates multiple, overlapping examples from one long text.*

So basically you have a window that can only see so many characters. We will call this our `block_size` (the maximum length our model can use as context). Once we place our window at the start of our text we can extract our first sequence. Then we move the window over by 1 character and extract the next sequence. We do this until we get to the end of our text.

### Processing in Parallel: Batches

Training each sequence separately would take too long. Our current technology allows for computations on sequences at a large scale. So if we only trained one at a time, we would not be taking full advantage of our computer's graphics processor (GPU).

To see how much batching matters, we can time it:

```text
$ python src/examples/ch12_batch_speed.py
32 sequences, one at a time : 2.191 s
the same 32 as one batch    : 0.245 s
batching is 9.0x faster on this machine
```

This is 9x faster. This is without using a GPU on your average computer. As seen from the log, the model in Chapter 13 trained for around 8 minutes on the computer that logged it. It would have taken more than an hour if we fed in one sequence at a time. With a GPU there is even more difference because there are thousands of processors available.

![Combining examples into a batch](https://jackluu.io/book/assets/diagrams/ch12-batching.png)

*Figure 12.3: A batch stacks multiple independent examples into a single block.*

We bundle many sequences together into a **batch** (Figure 12.3). Say our batch size is 32. We feed the model 32 independent sequences at once. The model runs them in parallel. It computes the loss across all 32 sequences and averages the loss.

### Why 32 and Not 1, or 1,000

There are two reasons why you want to batch. Speed is one. The other is how accurate your steps will be.

When we take our loss we get a step to adjust our weights. If our batch size were 1, we would be stepping in the direction that one snippet of Shakespeare tells us to. That could be a stage note or a long sequence of talk or maybe just blank space. We would jump in each direction and try to correct ourselves heavily each time. If we average 32 random chunks of Shakespeare together we get a more accurate step in the direction that the text wants to go as a whole.

A batch size of 1,000 would have two drawbacks. One being that all those sequences need to fit in memory. On a laptop, memory is the limit you hit first. You also wouldn't get much more accurate directions if you averaged 1000 snippets instead of 32 but your step would take 30x longer. There's nothing magic about a batch size of 32, and you'll see other books use 16 or 64. It's big enough to be accurate and small enough to fit.

Once we get to our "Training" step on our map we can begin to plug our batched data into our model and start the training loop.

## Code

To handle this efficiently, we use two built-in PyTorch tools: `Dataset` and `DataLoader` (Figure 12.4).

![Code flow: raw data into dataset and dataloader](https://jackluu.io/book/assets/diagrams/ch12-code-flow.png)

*Figure 12.4: The Dataset handles the sliding window, and the DataLoader stacks the examples into a batch.*

```python
class TextDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        # We need block_size + 1 tokens to form one (input, target) pair
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # Input is a block of text
        x = self.data[idx     : idx + self.block_size]
        # Target is the same text, shifted one character to the right
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

train_dataset = TextDataset(train_data, gpt_cfg.block_size)
val_dataset   = TextDataset(val_data,   gpt_cfg.block_size)

# DataLoader automatically batches the data for us
train_loader = DataLoader(
    train_dataset, batch_size=train_cfg.batch_size, shuffle=True
)
```

Run the full script to see the pipeline produce one batch:

```text
$ python src/ch11_dataloader.py
Data loaded: 1,115,394 total tokens
  Train : 1,003,854 tokens
  Val   : 111,540 tokens

Dataset sizes:
  Train examples: 1,003,726
  Val   examples: 111,412

DataLoader config:
  Batch size  : 32
  Train batches per epoch: 31,367

--- Inspecting one batch ---
x_batch shape: torch.Size([32, 128])  (batch_size, block_size)
y_batch shape: torch.Size([32, 128])  (batch_size, block_size)

First example in batch:
  x (input)  : 'ness! serious vanity!\nMis-shapen chaos of well-seeming for...
  y (target) : 'ess! serious vanity!\nMis-shapen chaos of well-seeming form...
  (y is x shifted by 1 character)

DataLoader ready! Ready for Chapter 13.
```

**What just happened:**

1.  We loaded our text file and converted it to token IDs.
2.  Lines 12 and 14 form the sliding window for `TextDataset`. Line 12 slices the text starting at position `idx`. Line 14 slices the same text shifted one character to the right, which is its target.
3.  Lines 21-23 create a `DataLoader`. It will automatically batch and shuffle our training data for us. When we go through all of the batches once, that is called an epoch, the term in the output above.
4.  We grabbed one batch and inspected it to confirm the target `y` is the input `x` shifted by one character.

**Shape Check:**

Table 12.1 lists the shapes of the batched input and target.

**Table 12.1:** Tensor shapes for the batched input and target sequences.

| Variable | Shape | Meaning |
| :--- | :--- | :--- |
| `x_batch` | `[32, 128]` | 32 sequences, each containing 128 input characters. |
| `y_batch` | `[32, 128]` | 32 sequences, each containing 128 target characters. |

## Try It

We can see the sliding window in action with a tiny dataset.

```python
"""Show how a sliding window creates multiple overlapping examples."""
import torch

data = torch.tensor([10, 20, 30, 40, 50, 60, 70])
block_size = 3

print(f"Data: {data.tolist()}")
print(f"Block size: {block_size}\n")

# A simple loop to show the sliding window
for i in range(len(data) - block_size):
    x = data[i : i + block_size]
    y = data[i + 1 : i + block_size + 1]
    print(f"Example {i+1}:")
    print(f"  Input  : {x.tolist()}")
    print(f"  Target : {y.tolist()}")
```

Lines 12 and 13 slice the array to create overlapping sequences for the input and target.

```text
$ python src/examples/ch12_batching_demo.py
Data: [10, 20, 30, 40, 50, 60, 70]
Block size: 3

Example 1:
  Input  : [10, 20, 30]
  Target : [20, 30, 40]
Example 2:
  Input  : [20, 30, 40]
  Target : [30, 40, 50]
Example 3:
  Input  : [30, 40, 50]
  Target : [40, 50, 60]
Example 4:
  Input  : [40, 50, 60]
  Target : [50, 60, 70]
```

> **Try It**
>
> Open `src/examples/ch12_batching_demo.py`. Change `block_size` to 4, run it again, and notice how the number of available examples decreases.
>

> **In Business**
>
> Imagine wanting your company's house-style assistant to learn from a library of documents. Say we had 100,000 company documents that we wanted to teach our assistant on. Obviously we're not teaching it one word at a time. When learning, you can teach your assistant in groups of 32 at a time. These could be emails or articles or even whole books. That is what makes it so efficient to teach your assistant on these large data sets.
>

> **Watch Out**
>
> Watch your dataset's `__len__`. If you have 100 characters and your `block_size` is 10, you only have 90 starting spots. Why? Because we need 11 characters to create a single example, 10 characters to input and 1 more to predict. That's why there's `len(self.data) - self.block_size` in the code.

## Key Takeaways

- A sliding window extracts overlapping sequences from a continuous block of text.
- Batching processes multiple independent sequences in parallel, which speeds up training.
- PyTorch's `Dataset` defines how to grab a single example.
- PyTorch's `DataLoader` handles the tedious work of grouping examples into batches and shuffling them.

## Check Your Understanding

1. If you have a sequence of 1000 tokens and a block size of 100, how many examples can a sliding window extract?
2. Why is batching important for training speed?
3. Why do we shuffle the training data?

## Assignment

> **Assignment 12: Follow one number**
>
> **About 15 minutes. You change one number and write no new code.** Work in this chapter's Colab notebook or on your own laptop.
>
> 1. **Run it.** Run `src/examples/ch12_batching_demo.py` as it is. Copy the four examples it prints, each with its Input and its Target.
> 2. **Change one number.** Find the line `data = torch.tensor([10, 20, 30, 40, 50, 60, 70])`. These seven numbers stand in for a text of seven characters. Change the `40` in the middle to any whole number that is not already in the list, and run the script again.
> 3. **Compare.** In how many examples is your number part of the Input? In how many is it part of the Target? How many examples does the script print now, and did that count change?
> 4. **Explain it.** In two or three sentences a manager could follow, say how a sliding window turns one text into many training examples, and why one character can appear in several of them.
>
> **Hand in:** both sets of four examples, your answers to step 3, and your sentences.

## Further Reading

**Why the field started building bigger.** Prior to this work, choices of model size, text volume to train on, and compute budget had been guesswork. This paper measured all three variables and discovered error decreases smoothly and predictably across an extremely wide range of model sizes. Turning model building into a cost/budget exercise explained why the industry then scaled up for the next few years. It also shows why there is a limit to how well the model you train yourself will do, because roughly 825,000 parameters and about a million characters of Shakespeare buy you a certain quality of text, no more.


Kaplan, J., McCandlish, S., Henighan, T., Brown, T. B., Chess, B., Child, R., Gray, S., Radford, A., Wu, J., & Amodei, D. (2020). *Scaling laws for neural language models* (arXiv:2001.08361). arXiv. https://doi.org/10.48550/arXiv.2001.08361

---

### `src/ch11_dataloader.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch11_dataloader.py"   # a cell has none, and the file uses it to find the text

"""
Create the Dataset and DataLoader for training.
This file belongs to Chapter 12.
Run: python src/ch11_dataloader.py
"""
import os
import sys
import torch
from torch.utils.data import Dataset, DataLoader


from src.utils.config import GPTConfig, TrainConfig

# Settings
gpt_cfg   = GPTConfig()
train_cfg = TrainConfig()

DATA_PATH = os.path.join(os.path.dirname(__file__), "data", "shakespeare.txt")

if not os.path.exists(DATA_PATH):
    print("ERROR: shakespeare.txt not found. Run download_data.py")
    sys.exit(1)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(set(text))
char_to_id = {ch: i for i, ch in enumerate(chars)}

# Store IDs in a PyTorch tensor
data  = torch.tensor([char_to_id[c] for c in text], dtype=torch.long)
n     = len(data)
split = int(0.9 * n)
train_data = data[:split]
val_data   = data[split:]

# --- The Idea ---
class TextDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        # We need block_size + 1 tokens to form one (input, target) pair
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # Input is a block of text
        x = self.data[idx     : idx + self.block_size]
        # Target is the same text, shifted one character to the right
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

train_dataset = TextDataset(train_data, gpt_cfg.block_size)
val_dataset   = TextDataset(val_data,   gpt_cfg.block_size)

# DataLoader automatically batches the data for us
train_loader = DataLoader(
    train_dataset, batch_size=train_cfg.batch_size, shuffle=True
)
val_loader = DataLoader(
    val_dataset, batch_size=train_cfg.batch_size, shuffle=False
)

# --- Demo ---
if __name__ == "__main__":
    torch.manual_seed(42)
    print("Chapter 12: Dataset and DataLoader\n")

    print(f"Data loaded: {n:,} total tokens")
    print(f"  Train : {len(train_data):,} tokens")
    print(f"  Val   : {len(val_data):,} tokens")

    print(f"\nDataset sizes:")
    print(f"  Train examples: {len(train_dataset):,}")
    print(f"  Val   examples: {len(val_dataset):,}")
    print(f"\nDataLoader config:")
    print(f"  Batch size  : {train_cfg.batch_size}")
    print(f"  Train batches per epoch: {len(train_loader):,}")

    print("\n--- Inspecting one batch ---")
    x_batch, y_batch = next(iter(train_loader))
    print(f"x_batch shape: {x_batch.shape}  (batch_size, block_size)")
    print(f"y_batch shape: {y_batch.shape}  (batch_size, block_size)")

    id_to_char = {i: c for i, c in enumerate(chars)}
    decode = lambda ids: "".join([id_to_char[i.item()] for i in ids])

    print(f"\nFirst example in batch:")
    print(f"  x (input)  : {repr(decode(x_batch[0]))[:60]}...")
    print(f"  y (target) : {repr(decode(y_batch[0]))[:60]}...")
    print(f"  (y is x shifted by 1 character)")

    print("\nDataLoader ready! Ready for Chapter 13.")

---

### `src/examples/ch12_batch_speed.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch12_batch_speed.py"   # a cell has none, and the file uses it to find the text

"""
Time 32 sequences run one at a time against the same 32 run as one batch.
This file belongs to Chapter 12.
Run: python src/examples/ch12_batch_speed.py
"""
import os
import sys
import time

import torch

from src.ch09_gpt_model import GPT
from src.utils.config import GPTConfig

cfg = GPTConfig()
torch.manual_seed(42)
model = GPT(cfg)
batch = torch.randint(0, cfg.vocab_size, (32, cfg.block_size))

with torch.no_grad():
    start = time.perf_counter()
    for row in batch:                       # one sequence at a time
        model(row.unsqueeze(0))
    one_at_a_time = time.perf_counter() - start

    start = time.perf_counter()
    model(batch)                            # all 32 together
    as_one_batch = time.perf_counter() - start

print(f"32 sequences, one at a time : {one_at_a_time:.3f} s")
print(f"the same 32 as one batch    : {as_one_batch:.3f} s")
print(f"batching is {one_at_a_time / as_one_batch:.1f}x faster on this machine")

---

### `src/examples/ch12_batching_demo.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch12_batching_demo.py"   # a cell has none, and the file uses it to find the text

"""Show how a sliding window creates multiple overlapping examples."""
import torch

data = torch.tensor([10, 20, 30, 40, 50, 60, 70])
block_size = 3

print(f"Data: {data.tolist()}")
print(f"Block size: {block_size}\n")

# A simple loop to show the sliding window
for i in range(len(data) - block_size):
    x = data[i : i + block_size]
    y = data[i + 1 : i + block_size + 1]
    print(f"Example {i+1}:")
    print(f"  Input  : {x.tolist()}")
    print(f"  Target : {y.tolist()}")